# CS156: Pipeline - First Draft

**Do I sound like AI, or have I always been a bad writer?**  

With the amount of AI we use on a day to day, I am fairly sure that a lot of the content I consume, whether it be the news, social media, or even the PCW that I grade, might be AI-assisted, if not completely AI-generated. This makes me wonder: as a symptom of consuming so much 'slop,' could my own writing also be showing traits commonly attributed to AI-generated text, such as structural uniformity, reduced variance in sentence length, or particular lexical patterns? 

This report answers this question by first training a model to classify assignments I wrote pre-Fall 2024, before AI tools had become sufficiently capable and convenient to function as routine academic aids, from later submissions. If the model can reliably distinguish between the two periods, this suggests measurable stylistic differences. I then examine which features drive this separation and analyze whether those differences align with documented stylistic characteristics often associated with AI-generated writing.

## The Data

I retrieved past written assignments from my Minerva email's Google Drive using a Colab notebook ([GitHub](https://colab.research.google.com/drive/1f_YmZ3cR82UPcLH7uNJ9DUu-x7Gllc7c#scrollTo=e160UdxwwBmt)), and manually labelled the documents I wanted to include.

### Exclusion Criteria

Since the focus is stylistic drift in *my* writing, I excluded several categories of assignments:

- **Skill builders from CS111 and CS113:** Mostly math and instructions, not much original prose. Also hand-written, which adds processing complexity.
- **Group assignments:** Written by multiple people, so I limited the dataset to me.
- **Heavily templated assignments:** Workbooks and worksheets with instructions included dilute my own voice. This excluded several FA50/FA51, Deep Dives from CS111/CS113, problem sets from CS114, and some SS110/SS152 assignments formatted as cover sheets.

I did include non-essay documents that reflect my own writing: video scripts and reflection documents from technical interviews.

Based on these criteria, 55 documents qualified. However, 4 were from outside Google Drive (PDFs from OverLeaf), so I excluded them to keep data processing consistent. That left 51 documents in the final dataset.

## Loading the Data

The scraped data is stored in a CSV file with UTF-8 encoding—I chose UTF-8 to preserve punctuation like em-dashes and curly quotes that might carry signal of style.

To set up the labels for my model, I extract course codes from the text itself using a regex pattern (`[A-Z]{2}\d{2,3}`) applied to the first 500 characters of each document. Since Minerva headers are reliably at the top, this captures the course code quickly.

Once I have the code, I map each document to a binary label:
- **0 (pre-AI):** Courses taken before Fall 2024 — CX51, EA51, FA51, GL92, MC51, CX50, EA50, FA50, GL91, MC50, IL199
- **1 (post-AI):** All other courses — everything taken from Fall 2024 onward, when LLMs like ChatGPT had reached mass adoption and I was actively using them in my work

This labeling scheme reflects the real-world timeline of when AI tools became standard in my academic toolkit.

## Loading the Data

Using the csv from scraped assigment data, I loaded it to a Pandas DataFrame with utf-8 encoding to preserve puctuation and letters. 

In [13]:
# Install necessary packages
!pip install pandas numpy scikit-learn matplotlib seaborn scipy textstat nltk spacy
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 26.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.5/124.5 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.6/770.6 kB 29.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 653.1/653.1 kB 24.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.8 MB/s eta 0:00:00
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 26.7 MB/s eta 0:00:00
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 5.3 MB/s eta 0:00:00a 0:00:01m
  Using cached charset_normalizer-3.4.4-cp311-cp311-macosx_10_9_universal2.whl (206 kB)
  Using cached idna-3.11-py3-none-any.whl (71 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  Using cached certifi-2026.1.4

In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
import spacy

# Sklearn imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Stylometric analysis
import textstat
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from collections import Counter


In [3]:
# Loading dataset

df = pd.read_csv("gdrive_scraped_data.csv", encoding='utf-8') 


## Pre-processing and EDA

While I am most interested in stylistic drift, I first train a baseline classifier (WHICH) to see whether there is any meaningful difference between my pre-AI (pre-Fall 2024) and post-AI (Fall 2024 and later) writing.

This section describes the exact cleaning steps used in the code and the assumptions behind them.

### Data Cleaning to Prevent Leakage

To ensure the classifier learns stylistic patterns rather than metadata, I wanted to train it on cleaned data where elements or sections that may cause leakage are removed. In order to compare model performance across data with different levels of cleaning to verify the model is truly learning from the main text, I created a fixed cleaning pipeline: `remove_cover_sheet` -> `remove_toc` -> `extract_body_only` -> `remove_hc_tags`.

- `text_raw`: Original text with all content. This is pulled directly from the source csv file. 
- `text_clean`: Fully cleaned. This removes the cover sheet, TOC, References, Appendices, and HC/LO tags, which can cause leakage from course/topic/time. I also removed AI Statements because the wording, tool names, and format may shift over time or by course. This is justified as the goal is to capture main-body prose, not these sections, that are less stylistic and more structural. 

I still use an intermediate `no_cover` step inside the pipeline (cover sheet + TOC removed), because the heuristics build on each other, but I decided not to keep it as a permanent column in the final dataframe.

In addition to cleaning, I added the following metadata columns: 
- `course_code`: Extracted from the cover sheet using a 2-letter + 2-3 digit regex. 
- `label`: The label for model training. Pre-AI work is represented by 0 and post-AI work is represented by 1. 

Importantly, each cleaning step makes some key assumptions about my data, which I verified as I implemented the code: 

- **Cover sheets or headers exist at the start of the document.** The cover sheet is assumed to appear in the first 500-3000 characters and to contain either a Minerva header, a horizontal rule, a large page break, or a course code. This lets the heuristic remove only the top portion.
- **TOCs are near the top and look like lists with page numbers.** The TOC removal checks the first ~20 lines for multiple lines ending in digits or an explicit "Contents" header.
- **End-matter starts at explicit markers.** The body is truncated at the earliest of: References/Bibliography, Appendix/Appendices, Word Count, or AI Statement/AI Use headings, but only if they appear after 100+ characters.
- **HC/LO tags are structured as #tag: blocks.** The cleaning assumes these are metadata blocks that can be removed without losing core prose.

This setup lets me compare model performance on raw text vs. main-body prose to check whether the model is learning writing style rather than metadata.


In [16]:
# Preprocessing functions

VALID_COURSES = ['CP192', 'CS146', 'GL96', 'CS156', 'SS111',
                 'CP191', 'CS166', 'GL95', 'SS152', 'SS156', 
                 'CS113', 'CS114', 'GL94', 'SS110', 
                 'CS110', 'CS111', 'GL93', 'SS112', 
                 'CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                 'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']

def extract_course_code(text):
    """
    Extract course code from cover sheet (e.g., CS110, GL96).
    
    Searches first 500 chars for 2-letter prefix + 2-3 digit number.
    Returns: course_code or None if not found
    """
    pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    matches = re.findall(pattern, text[:500])
    
    if matches:
        prefix, number = matches[0]
        return f"{prefix}{number}"
    
    return None

def remove_cover_sheet(text):
    """
    Remove cover sheet content using layered heuristics.
    
    Strategies (applied in order):
    1) Minerva-specific: Searches for horizontal rules or TOC header in first 3000 chars
    2) Major page break: 6+ consecutive blank lines
    3) Early horizontal rule: 5+ underscores in first 2000 chars
    4) Course code fallback: Skips 2 lines after finding course code
    """
    text = text.replace('\r\n', '\n')

    if 'Minerva University' in text[:500]:
        early_section = text[:3000]
        rule_match = re.search(r'\n\s*_{5,}\s*\n', early_section)
        toc_match = re.search(r'(?i)\n\s*(table of )?contents\s*\n', early_section)
        candidates = []
        if rule_match:
            candidates.append(('rule', rule_match.start(), rule_match.end()))
        if toc_match:
            candidates.append(('toc', toc_match.start(), toc_match.end()))
        if candidates:
            kind, start, end = min(candidates, key=lambda x: x[1])
            return text[start:] if kind == 'toc' else text[end:]

    page_break_pattern = r'(\n\s*){6,}'
    match = re.search(page_break_pattern, text)
    if match:
        return text[match.end():]

    horizontal_rule_pattern = r'\n\s*_{5,}\s*\n'
    match = re.search(horizontal_rule_pattern, text[:2000])
    if match:
        return text[match.end():]

    course_code_pattern = r'\b([A-Z]{2})[\s-]?(\d{2,3})\b'
    match = re.search(course_code_pattern, text[:500])
    if match:
        pos = match.end()
        for _ in range(2):
            next_newline = text.find('\n', pos)
            if next_newline == -1:
                break
            pos = next_newline + 1
        return text[pos:]

    return text

def remove_hc_tags(text):
    """
    Remove HC tags (#tag: explanation) and standalone horizontal rules.
    
    Removes:
    - HC tag blocks: #tag: explanation (continues until blank line)
    - Horizontal rules: Lines with 5+ underscores
    """
    text = re.sub(r'\#[A-Za-z0-9\-]+:[^\n]*(?:\n(?!\n)[^\n]*)*', '', text)
    text = re.sub(r'^\s*_{5,}\s*$', '', text, flags=re.MULTILINE)
    return text

def remove_toc(text):
    """
    Remove table of contents blocks using two strategies.
    
    Strategy 1 - Explicit header: Matches "Table of Contents" or "Contents" header.
                 Ends at double newline or horizontal rule.
    Strategy 2 - Page number heuristic: Detects 3+ lines ending with digits in first 20 lines.
                 Removes up to first double blank line, or first blank line as fallback.
    """
    text = text.replace('\r\n', '\n')

    toc_pattern = r'(?i)(table of )?contents\s*\n'
    match = re.search(toc_pattern, text)
    if match:
        toc_start = match.start()
        remaining = text[match.end():]
        end_match = re.search(r'\n\s*\n|\n\s*_{5,}\s*\n', remaining)
        if end_match:
            return text[:toc_start] + remaining[end_match.end():].lstrip()
        return text[:toc_start]

    lines = text.split('\n')
    sample = lines[:20]
    toc_like = [ln for ln in sample if re.search(r'\s+\d+\s*$', ln)]
    if len(toc_like) >= 3:
        for i in range(len(sample) - 1):
            if sample[i].strip() == '' and sample[i + 1].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()
        for i in range(len(sample)):
            if sample[i].strip() == '':
                return '\n'.join(lines[i + 1:]).lstrip()

    return text

def extract_body_only(text):
    """
    Extract main body text by cutting at the earliest end-of-body marker.
    
    Detects: References, Bibliography, Appendix, Appendices, Word Count, AI Statement variants.
    Protection: Only cuts if marker appears after 100+ chars (prevents removing entire document).
    """
    text = text.replace('\r\n', '\n')
    
    end_markers = [
        r'(?i)^(references|bibliography)',
        r'(?i)^(appendix|appendices)',
        r'(?i)^(word count|ai statement|ai use|statement on ai)',
    ]
    
    earliest_match = None
    earliest_pos = len(text)
    
    for pattern in end_markers:
        match = re.search(pattern, text, re.MULTILINE)
        if match and match.start() > 100 and match.start() < earliest_pos:
            earliest_pos = match.start()
            earliest_match = match
    
    if earliest_match:
        return text[:earliest_match.start()].strip()
    
    return text.strip()


def preprocess_dataframe(df):
    """
    Apply all preprocessing steps to create cleaned text columns.
    
    Columns created:
    - text_raw: Original text (baseline)
    - text_clean: Body text only (no coversheet, TOC, references, appendix, AI statements, HC tags)
    - course_code: Extracted course code
    - label: 0 = pre-AI courses, 1 = post-AI courses
    """
    df_processed = df.copy()
    df_processed['course_code'] = df_processed['text_raw'].apply(extract_course_code)
    df_processed['no_cover'] = df_processed['text_raw'].apply(remove_cover_sheet).apply(remove_toc)
    df_processed['text_clean'] = df_processed['no_cover'].apply(extract_body_only).apply(remove_hc_tags)
    PRE_AI_COURSES = ['CX51', 'EA51', 'FA51', 'GL92', 'MC51', 
                      'CX50', 'EA50', 'FA50', 'GL91', 'MC50', 'IL199']
    df_processed['label'] = df_processed['course_code'].apply(
        lambda x: 0 if x in PRE_AI_COURSES else 1
    )
    
    return df_processed

# Apply preprocessing and keep relevant columns
df_processed = preprocess_dataframe(df)
df_processed = df_processed[['name', 'course_code', 'label', 'text_raw', 'text_clean']]

print("Preprocessing complete.")
print(f"Total documents: {len(df_processed)}")

# Display summary table with truncated text for readability
display_df = df_processed.head().copy()
text_cols = ['text_raw', 'text_clean']
for col in text_cols:
    display_df[col] = display_df[col].apply(lambda x: (x[:75] + '...') if isinstance(x, str) and len(x) > 75 else x)
display(display_df[['course_code', 'name', 'label'] + text_cols])

Preprocessing complete.
Total documents: 51


,course_code,name,label,text_raw,text_clean
0,CP192,Reflection on Track Options [final],1,﻿Reflection on Track Options\r\n\r\n\r\n\r\n\r\nMinerva University\r\nCP192: Capstone S...,Reflection on Track Options\nProcess Documentation\nTo ideate what kind of tr...
1,SS111,LBA [final],1,﻿LBA: Analyzing the Bangle Market of Charminar\r\n\r\n\r\n\r\n\r\nMinerva University\r...,LBA: Analyzing the Bangle Market of Charminar\n Located in the “Old C...
2,GL96,Elevation Reflection & Engagement [final],1,﻿Elevation Reflection & Engagement \r\nGL96\r\nSuisei Nakagawa\r\n\r\n\r\nOne way I p...,One way I plan to engage meaningfully with Hyderabad is through its local c...
3,GL96,Ethnographic Assignment [final],1,﻿Ethnographic Assignment\r\n\r\n\r\n\r\n\r\nMinerva University\r\nGL96: Global Learning...,"Ethnographic Assignment\nIntroduction\nFor this assignment, I spoke to Arjun ..."
4,SS156,Final assignment [final],1,﻿Tab 1\r\n\r\n\r\n\r\n\r\n\r\n\r\nComparative Analysis of Political Systems: Furusao Noze...,Comparative Analysis of Political Systems: Furusao Nozei in Japan\nIntroduct...


In [ ]:
# # Manual inspection of all documents
# # Commented out to reduce bulk

# pd.set_option('display.max_colwidth', None)

# for idx, row in df_processed.iterrows():
#     print(f"\n{'='*100}")
#     print(f"📄 Document #{idx + 1}: {row['name']}")
#     print(f"   Course: {row['course_code']} | Label: {row['label']} ({'pre-AI' if row['label'] == 0 else 'post-AI'})")
#     print(f"{'='*100}")
        
#     print(f"\n[TEXT_CLEAN - First 100 chars]")
#     print(row['text_clean'][:100] if isinstance(row['text_clean'], str) else row['text_clean'])
#     print(f"\n[TEXT_CLEAN - Last 100 chars]")
#     print(row['text_clean'][-100:] if isinstance(row['text_clean'], str) else row['text_clean'])


📄 Document #1: Reflection on Track Options [final]
   Course: CP192 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
Reflection on Track Options
Process Documentation
To ideate what kind of track to propose, I answere

[TEXT_CLEAN - Last 100 chars]
 meetings, as is the case in traditional capstone advising.
HC/LO Applications for the Work Product


📄 Document #2: LBA [final]
   Course: SS111 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
LBA: Analyzing the Bangle Market of Charminar
        Located in the “Old City” of Hyderabad, India,

[TEXT_CLEAN - Last 100 chars]
s firms restrict output below the efficient level in order to sustain prices above marginal cost.[3]

📄 Document #3: Elevation Reflection & Engagement [final]
   Course: GL96 | Label: 1 (post-AI)

[TEXT_CLEAN - First 100 chars]
One way I plan to engage meaningfully with Hyderabad is through its local climbing community. I bega

[TEXT_CLEAN - Last 100 chars]
e me to pay attention to differences rather than projec

### Text Normalization and Tokenization

In addition to separating main text from structural artefacts, I standardize case and whitespace to reduce surface-level variation (e.g., casing and uneven spacing) before tokenization using spaCy. I chose spaCy over a simple regex-based tokenizer because this analysis depends on consistent sentence boundaries and stable token definitions. Naive regex rules tend to break on abbreviations, decimal numbers, ellipses, and formatting irregularities, which would directly distort structural metrics of interest like mean sentence length and variance. 

I disable part-of-speech tagging, dependency parsing, named entity recognition, and lemmatization to keep the pipeline lightweight and deterministic. In particular, lemmatization would collapse inflected word forms and remove variation that may itself be stylistically meaningful.

This step produces three representations per document: a list of tokens (including punctuation), a word list excluding whitespace, and a list of sentence strings. These structured outputs form the basis for feature engineering and exploratory analysis.

In [21]:
# Normalization 

def normalize_text(text):
    """Lowercase and collapse whitespace for consistent tokenization."""
    if not isinstance(text, str):
        return text
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Create a normalized column from the raw and cleaned text
df_processed["text_raw_norm"] = df_processed["text_raw"].apply(normalize_text)
df_processed["text_clean_norm"] = df_processed["text_clean"].apply(normalize_text)

print("Normalization complete.")

# Display summary table with truncated text for readability
display_df = df_processed.head().copy()
text_cols = ['name', 'text_raw_norm', 'text_clean_norm']
for col in text_cols:
    display_df[col] = display_df[col].apply(lambda x: (x[:75] + '...') if isinstance(x, str) and len(x) > 75 else x)
display(display_df[['course_code', 'name', 'label'] + text_cols])


Normalization complete.


,course_code,name,label,name,text_raw_norm,text_clean_norm
0,CP192,Reflection on Track Options [final],1,Reflection on Track Options [final],﻿reflection on track options minerva university cp192: capstone seminar pro...,reflection on track options process documentation to ideate what kind of tr...
1,SS111,LBA [final],1,LBA [final],﻿lba: analyzing the bangle market of charminar minerva university ss111: mo...,lba: analyzing the bangle market of charminar located in the “old city” of ...
2,GL96,Elevation Reflection & Engagement [final],1,Elevation Reflection & Engagement [final],﻿elevation reflection & engagement gl96 suisei nakagawa one way i plan to e...,one way i plan to engage meaningfully with hyderabad is through its local c...
3,GL96,Ethnographic Assignment [final],1,Ethnographic Assignment [final],﻿ethnographic assignment minerva university gl96: global learning january 9...,"ethnographic assignment introduction for this assignment, i spoke to arjun ..."
4,SS156,Final assignment [final],1,Final assignment [final],﻿tab 1 comparative analysis of political systems: furusao nozei in japan mi...,comparative analysis of political systems: furusao nozei in japan introduct...


In [25]:
# SpaCy tokenizer 

nlp = spacy.load("en_core_web_sm", disable=["ner", "tagger", "parser", "lemmatizer"])
nlp.add_pipe("sentencizer")

def spacy_tokenize(text):
    """
    Tokenize text with spaCy.

    Returns a dict with:
    - tokens: all tokens including punctuation
    - words: tokens excluding spaces
    - sentences: list of sentence strings
    """
    if not isinstance(text, str):
        return {"tokens": [], "words": [], "sentences": []}

    doc = nlp(text)
    tokens = [token.text for token in doc]
    words = [token.text for token in doc if not token.is_space]
    sentences = [sent.text.strip() for sent in doc.sents]

    return {"tokens": tokens, "words": words, "sentences": sentences}

df_processed["text_raw_tokens"] = df_processed["text_raw_norm"].apply(spacy_tokenize)
df_processed["text_clean_tokens"] = df_processed["text_clean_norm"].apply(spacy_tokenize)
print("Tokenization complete.")


Tokenization complete.


### Feature Engineering

### EDA

## Analysis

## Model Selection 

## Training 

In [6]:
#@title also session 5 PCW (CHANGE LATER)

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics, naive_bayes

labels, texts = df['v1'], df['v2']
vectorizer = text.TfidfVectorizer()
vec_texts = vectorizer.fit_transform(texts)


# We train a model
model = naive_bayes.MultinomialNB()
model.fit(vec_texts, labels)
pred_labels = model.predict(vec_texts)

confusion_mat = metrics.confusion_matrix(labels, pred_labels)
sns.heatmap(confusion_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# some checks to see how the model is doing
print(model.classes_)
print(df['v1'].value_counts())
print(vec_texts.shape) 




KeyError: 'v1'

## Predictions

## Discussion 

## Summary 

## References 

- Session 5 PCW: https://forum.minerva.edu/app/courses/3804/sections/13026/classes/99436 